# LLM Conversation Keywords Extractor

In [17]:
import json
import spacy
from collections import Counter
import unicodedata
from typing import List
from role_enum import Role

nlp = spacy.load("en_core_web_sm", disable=["ner"])

def normalize(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    return text


def extract_top_keywords(
    json_path: str,
    roles: List[Role],
    top_n: int = 20,
    lemmatize: bool = False
):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    texts = []
    messages = data.get("messages", [])

    allowed_roles = {r.value for r in roles}

    for msg in messages:
        if msg.get("role") in allowed_roles:
            content = msg.get("content", "")
            if content:
                texts.append(normalize(content))

    all_tokens = []

    for doc in nlp.pipe(texts):
        for token in doc:
            if (
                not token.is_stop and
                not token.is_punct and
                token.is_alpha and
                len(token.text) > 2
            ):
                word = token.lemma_.lower() if lemmatize else token.text.lower()
                all_tokens.append(word)

    counter = Counter(all_tokens)

    result = [
        {"keyword": word, "frequency": freq}
        for word, freq in counter.most_common(top_n)
    ]

    return result

In [25]:
keywords = extract_top_keywords("data/sample.json", roles=[Role.USER, Role.ASSISTANT], lemmatize=True)

display(keywords)

[{'keyword': 'keyword', 'frequency': 9},
 {'keyword': 'base', 'frequency': 7},
 {'keyword': 'extraction', 'frequency': 6},
 {'keyword': 'language', 'frequency': 5},
 {'keyword': 'nlp', 'frequency': 5},
 {'keyword': 'like', 'frequency': 5},
 {'keyword': 'system', 'frequency': 5},
 {'keyword': 'processing', 'frequency': 4},
 {'keyword': 'machine', 'frequency': 4},
 {'keyword': 'use', 'frequency': 4},
 {'keyword': 'include', 'frequency': 4},
 {'keyword': 'text', 'frequency': 4},
 {'keyword': 'conversation', 'frequency': 4},
 {'keyword': 'idf', 'frequency': 4},
 {'keyword': 'textrank', 'frequency': 4},
 {'keyword': 'approach', 'frequency': 4},
 {'keyword': 'method', 'frequency': 4},
 {'keyword': 'yes', 'frequency': 4},
 {'keyword': 'human', 'frequency': 3},
 {'keyword': 'application', 'frequency': 3}]